# Learning Rate

Here we are discussing about selecting Learning rate for training the model. For selecting the Learning rate the usual technique is trial and error where we test multiple learning rates and select the best. 
Other issue is that learning rate suitable for different optimizers are different. So if we change the optimizer then again we have to search what is the best learning rate.  

Commonly used lr for Adam Optimizer is 3e-4(known as Karpathy's constant).  

How ever a technique for this has been brought up by Jeremy Howard in his fast.ai course.  

Idea of it is as follows.  
 - start with a small learning rate and increase it to a higher value over mini batches in an epoch.  
 - Calculate the loss for each rate and then select the value where loss has steepest decline.(Where reducing rate of loss is highest)

Let's have a look at the implementaion of this algorithm.

In [ ]:
import math
train_loader = None

def find_lr(model, loss_fn, optimizer, init_val=1e-8, final_val=1):
    number_in_epoch = len(train_loader-1)
    update_step = (final_val / init_val)**(1/number_in_epoch)
    lr = init_val
    optimizer.param_groups[0]["lr"] = lr
    best_loss = 0.0
    batch_num = 0
    losses = []
    log_lrs = []

    for data in train_loader:
        batch_num+=1
        inputs,labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs,labels)

        if batch_num > 1 and loss > 4*best_loss:
            return log_lrs[10:-5], losses[10:-5]

        # Record best loss

        if loss < best_loss or batch_num == 1:
            best_loss = loss

        losses.append(loss)
        log_lrs.append(math.log10(lr))

        lr*=update_step
        optimizer.param_groups[0]['lr'] = lr
    
    return log_lrs[10:-5], losses[10:-5]

In [ ]:
import matplotlib.pyplot as plt


log,losses = find_lr()
found_lr = 1e-2
plt.plot(log,losses)

When building models from scratch using a const. lr for all the layers makes sense. But when it comes to transfer learning we can get better results by training different layers at different rates. We can implement it as follows in pytorch

In [ ]:
import torch.optim as optimizer
from torchvision import models


transfer_model = models.resnet50(weights=models.ResNet50_Weights)

optimizer = optimizer.Adam([
    {'params' : transfer_model.layer4.parameters(), 'lr' : found_lr/3},
    {'params' : transfer_model.layer4.parameters(), 'lr' : found_lr/9},
    ], lr = found_lr)
